# Jam Transformer — Colab Smoke Test

Google Drive에 올려둔 번들에서 코드를 받아와 **합성 데이터**로 파이프라인 전체를 빠르게 확인합니다.  
실제 데이터 없이도 동작하며 T4 기준 **2~3분** 안에 완료됩니다.

| 단계 | 내용 |
|---|---|
| 0 | GPU 확인 |
| 1 | Drive 마운트 + 번들 추출 |
| 2 | 의존성 설치 |
| 3 | 합성 데이터 생성 + 학습 (2 epoch) |
| 4 | 추론 — 합성 멜로디 → 반주 생성 |

> **번들 준비 (로컬 PC):**
> ```powershell
> # 소스 코드만 (~1 MB, 데이터 제외)
> docker compose run --rm train python scripts/package_for_server.py `
>   --include_data "" --compress gz --out bundles/jam_tx_src.tgz
> ```
> `bundles/jam_tx_src.tgz` 를 Google Drive에 업로드하세요.
>
> **런타임 설정:** `런타임 > 런타임 유형 변경 > T4 GPU` 선택 후 실행하세요.

In [ ]:
# ============================================================
#  여기만 수정하세요
# ============================================================

# Google Drive 상의 번들 경로 (소스 코드 전용 or 전체 번들 모두 사용 가능)
DRIVE_BUNDLE = '/content/drive/MyDrive/jam_tx_src.tgz'

# ============================================================

## 0. GPU 확인

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0 or not result.stdout.strip():
    raise RuntimeError(
        "GPU가 없습니다. [런타임 > 런타임 유형 변경 > T4 GPU] 로 변경 후 재실행하세요."
    )
print("GPU:", result.stdout.strip())

import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.version.cuda}  |  "
      f"device: {torch.cuda.get_device_name(0)}")

## 1. Google Drive 마운트 + 번들 추출

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Drive 마운트 완료")

In [ ]:
import os, shutil, subprocess, time, sys
from pathlib import Path

PROJECT_DIR = Path('/content/project_transformer')
local_bundle = Path('/content/jam_tx_src.tgz')

if not PROJECT_DIR.exists():
    print(f'Drive에서 번들 복사 중... ({DRIVE_BUNDLE})')
    t0 = time.time()
    shutil.copy(DRIVE_BUNDLE, local_bundle)
    print(f'  복사 완료: {local_bundle.stat().st_size/1e6:.1f} MB  '
          f'({time.time()-t0:.1f}s)')

    print('압축 해제 중...')
    t0 = time.time()
    subprocess.run(['tar', 'xzf', str(local_bundle), '-C', '/content/'], check=True)
    print(f'  완료 ({time.time()-t0:.1f}s)  ->  {PROJECT_DIR}')
    local_bundle.unlink()
else:
    print(f'{PROJECT_DIR} 이미 존재 — 압축 해제 건너뜀')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print(f'CWD: {os.getcwd()}')

## 2. 의존성 설치

PyTorch / CUDA는 Colab에 이미 설치되어 있으므로 재설치하지 않습니다.

In [ ]:
%%bash
pip install -q \
    pytorch-lightning>=2.2 \
    miditoolkit \
    loguru \
    pretty_midi \
    pyyaml \
    python-dotenv

# 패키지 자체 설치 (torch 재설치 없이)
pip install -q -e . --no-build-isolation 2>&1 | tail -3
echo "설치 완료"

## 3. 합성 데이터 생성 + 학습 (2 epoch)

실제 MIDI 데이터 없이 합성 데이터로 파이프라인 전체를 검증합니다.  
2 epoch 완료까지 T4 기준 **1~2분** 소요됩니다.

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'

# 합성 데이터 생성 (32곡)
!python scripts/prepare_data.py --synthetic --num_songs 32 --out_dir data/smoke_test

In [ ]:
# compile=false 는 config 기본값이므로 별도 설정 불필요
!python scripts/train.py \
    --data_dir data/smoke_test \
    --epochs 2 \
    --set training.batch_size=4 \
    --set training.val_batch_size=4 \
    --set training.early_stopping_enabled=false \
    --set training.log_to_file=false \
    --set training.csv_logger_enabled=false

## 4. 추론 — 합성 멜로디 → 반주 생성

학습 초기라 생성 품질은 좋지 않습니다.  
중요한 것은 **파이프라인 전체가 오류 없이 동작하는지** 입니다.

In [ ]:
from pathlib import Path
from jam_transformer.config import load_config
from jam_transformer.midi_io import events_to_midi
import sys
sys.path.insert(0, 'scripts')
from prepare_data import _synthesize_song

cfg = load_config('configs/config.yaml')
# _synthesize_song 반환: (events, tempo, key_root, key_mode)
events, tempo, _kr, _km = _synthesize_song(seed=999)
melody_only = [e for e in events if e.track == 'melody']
midi = events_to_midi(melody_only, cfg.tokenizer, tempo_bpm=tempo)
Path('output').mkdir(exist_ok=True)
midi.dump('output/prompt_melody.mid')
print('Wrote output/prompt_melody.mid')

In [ ]:
import glob, os, subprocess, sys

ckpts = sorted(glob.glob('checkpoints/**/*.ckpt', recursive=True), key=os.path.getmtime)
if not ckpts:
    raise FileNotFoundError('체크포인트가 없습니다. 학습 셀을 먼저 실행하세요.')
ckpt = ckpts[-1]
print(f'사용할 체크포인트: {ckpt}')

result = subprocess.run([
    sys.executable, 'scripts/inference.py',
    '--checkpoint', ckpt,
    '--melody_midi', 'output/prompt_melody.mid',
    '--output',      'output/accompaniment.mid',
    '--temperature', '1.0',
    '--top_k', '64', '--top_p', '0.95',
    '--max_new_tokens', '256',
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('inference.py 실패')
print('추론 완료!')

In [ ]:
import miditoolkit

m = miditoolkit.MidiFile('output/accompaniment.mid')
total_notes = 0
for inst in m.instruments:
    n = len(inst.notes)
    total_notes += n
    if n > 0:
        print(f'[{inst.name}]  {n} notes  '
              f'first5={[note.pitch for note in inst.notes[:5]]}')

print(f'\n총 생성 노트 수: {total_notes}')
if total_notes == 0:
    print('WARNING: 노트 없음. max_new_tokens를 늘리거나 temperature를 높여보세요.')
else:
    print('\n✓ 파이프라인 정상 동작 확인!')